In [ ]:
#| default_exp game/food

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay
from HexMagic.water.soil import SoilSystem

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import math
from dataclasses import dataclass, field

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep


In [ ]:
myTerr = TerraDemo().japan_korea_map()
rivers = myTerr.carve_to_ocean(num_lakes=0)

In [ ]:
myTerr.compute_climate()

In [ ]:
basins = DrainageBasins(myTerr)

In [ ]:
#TerrainDisplay(TerrainOverlay(),ClimateOverlay(),terrain=myTerr)

In [ ]:
!cat ../../HexMagic/water/*.py

In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.ROOK, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()

So we want to be able to compute yeilds of food per turn. I am thinking of 8 tiers.

I want crop yield to be a combination of flow, temperature, and possibly soil.

we are going to want to create a dotted overlay base upon yeild similar to our climate overlay.

lets work throught the design

I think the temerature and flow are relatively easy to get. Is soil going to make a difference? should we keep it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good question. Looking at your existing `SoilSystem`, the 5 types already have properties that map nicely to fertility:

| Soil | Permeability | Real-world farming | Yield multiplier? |
|------|-------------|--------------------|--------------------|
| Granite | 0.2 | Terrible — bedrock | ~0.2 |
| Basalt | 0.3 | Poor — volcanic rock | ~0.3 |
| Limestone | 0.6 | Decent — karst grasslands | ~0.7 |
| Sandstone | 0.7 | Moderate — drains well | ~0.6 |
| **Alluvial** | 0.4 | **Excellent** — river sediment | ~1.0 |

The case **for** keeping it: alluvial soil already concentrates near river mouths and coasts (your `from_plates` logic assigns it at `elev < elev_threshold` near coast). So you'd get the historically accurate pattern of *river deltas = breadbaskets* emerging naturally — Nile, Mesopotamia, etc. That's a satisfying emergent property.

The case **against**: soil already *correlates* with flow + low elevation. If you have high flow + warm temperature, you're probably already in a fertile lowland with alluvial soil. So it might not add much *distinction* beyond what flow × temperature gives you.

My take: **keep it, but make it dead simple** — just a single `fertility` float per `SoilType` used as a multiplier. It's one extra field on a dataclass you already have, `soil_type` is already in `terrain.fields`, and it breaks the tie between "rocky mountain valley with a stream" vs "alluvial floodplain" in a way that feels right. No new computation needed.

The yield formula would then be something like:

```
yield_tier = f(flow, temperature, soil_fertility)
```

where soil is just a `0.2–1.0` multiplier. Cheap and flavorful.

Sounds great. so is there a way to calculate how much food you can grow on a tile?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, there are well-established real-world models we can borrow from. The FAO's Agro-Ecological Zones approach boils down to roughly what we already discussed — three factors multiplied together:

**Temperature factor** — bell curve, peak around 20°C:
- < 0°C → 0 (frozen)
- 5–10°C → 0.3 (short growing season, hardy crops only)
- 15–25°C → 0.8–1.0 (sweet spot for most staples)
- 30–35°C → 0.5 (heat stress, need irrigation)
- \> 40°C → 0.1 (barely anything grows)

**Water factor** — precipitation + flow, with diminishing returns:
- Near-zero → 0.1 (desert)
- Moderate rain → 0.5–0.7
- Good rain + river flow → 0.9–1.0
- Excessive (flooded) → drops back to 0.5

**Soil fertility** — the multiplier we just discussed (0.2–1.0).

Then:

```python
raw_yield = temp_factor * water_factor * soil_fertility
tier = int(raw_yield * 7)  # 0-7 → 8 tiers
```

You already have everything you need in `terrain.fields`:
- Temperature comes from `compute_climate()` (your climate zones map to temp ranges)
- Water comes from `precipitation` + `flow`
- Soil from `soil_type`

The nice thing is this naturally produces the right patterns — river valleys in temperate zones become tier 7 breadbaskets, arctic mountains become tier 0, and deserts sit around tier 1-2. No special cases needed.

Want to start with the temperature curve, since your climate zones already give us temperature bands to work from?

How much of this can you build? I think we need to experiment a bit.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

I can build the whole computation + overlay in one go. Here's a starting point you can tweak:

```python
import numpy as np
import math
from dataclasses import dataclass, field

@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve: dict = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = raw.max() if raw.max() > 0 else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")
```

And to test it:

```python
# Make sure soil exists
from HexMagic.water.soil import SoilSystem
soil = SoilSystem.from_plates(myTerr, [])

fy = FoodYield(myTerr, basins)
fy.compute()
fy.summary()
```

Run that and let's see what the distribution looks like — then we can tune the curves and build the overlay once we're happy with the numbers.

In [ ]:


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve: dict = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
# Make sure soil exists
from HexMagic.water.soil import SoilSystem
soil = SoilSystem.from_plates(myTerr, [])

fy = FoodYield(myTerr, basins)
fy.compute()
fy.summary()


I think this is ok. there is a bunch of mountains and oceans

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, that actually tracks well. Japan is only about 12% arable land in reality — your tiers 4+ total ~455/2467 ≈ 18%, which is in the right ballpark for a mountainous coastal map.

One thing to note: the jump from tier 5 (288) to tier 6 (6) is a bit steep. That's likely because the normalization divides by the single highest hex, so only a handful of "perfect" hexes reach the top tiers. We could soften that by normalizing to the 95th percentile instead of the max:

```python
# Replace this line in compute():
rmax = raw.max() if raw.max() > 0 else 1.0

# With:
rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
```

That would spread the top tiers out more — the very best hexes would all land at tier 7, but you'd get a smoother gradient into tiers 5-6. Try it and see if the distribution feels better?

Then once we're happy with the numbers, we can build the dotted overlay.

In [ ]:
fy = FoodYield(myTerr, basins)
fy.compute()
fy.summary()

Thoughts

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Much better gradient through the middle tiers. The shape tells a good story:

- **Tier 0 (1114)** — mountains, tundra, bare rock. ~45% of land. Realistic for Japan/Korea.
- **Tiers 1-6 (1031)** — nice smooth gradient, no cliffs. This is the "interesting" range for gameplay.
- **Tier 7 (322)** — river valleys and alluvial lowlands. ~13% of land.

Tier 7 is chunky because everything above the 95th percentile gets clamped there. But honestly, for a game that's probably a feature — those are the breadbasket hexes players fight over, and you *want* enough of them to support settlements.

If it bugs you, you could try the 90th percentile to squeeze it down, but I think this is solid enough to move to the overlay. Ready to build it?

lets build the overlay. I know we need terrainPatterns for the dots. What would be a good color to use. I am assuming we will use it either with a cream backgroun or the elevations. does it make sense to have different colors or just different densities

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

For an overlay that sits on top of terrain, **single color + varying density** is the way to go. Two reasons:

1. **Readability** — density alone gives you 8 distinct levels without competing with the base map's color palette. Multiple colors on top of elevation colors would be a mess.
2. **Consistency** — it's the same language as your `dotted_watershed_overlay`, so users learn one visual system.

For the color itself, I'd go with a **warm earthy green** — something like:

- `#558B2F` (olive green) — earthy, reads as "farmland," won't clash with ocean blues or mountain browns
- `#2E7D32` (forest green) — slightly cooler, more contrast on cream
- `#8D6E63` (warm brown) — if you want "wheat fields" rather than "crops"

I'd lean toward **`#558B2F`** — it's muted enough to overlay without screaming, warm enough to feel like agriculture, and distinct from the blue river/water system.

Here's the overlay builder:

```python
from HexMagic.terrainpatterns import TerrainPatterns

def food_overlay(fy: FoodYield, color: str = "#558B2F") -> str:
    """Build dotted overlay from computed food yield tiers."""
    terrain = fy.terrain
    grid = terrain.hexGrid
    
    patGen = TerrainPatterns(terrain)
    patterns = patGen.ballDensity(
        levels=fy.n_tiers,
        fills=[color],
        prefix="food_yield"
    )
    
    overlay = ""
    used = set()
    
    for i in range(len(terrain.elevations)):
        if terrain.elevations[i] <= 0:
            continue
        tier = int(fy.tiers[i])
        if tier == 0:
            continue  # Skip barren — let base map show through
        
        used.add(tier)
        pat_name = patterns[tier].attributes['id']
        hex_obj = grid.hexes[i]
        
        pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
        overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
    
    for tier in sorted(used):
        grid.builder.add_definition(patterns[tier])
    
    return overlay
```

Try it with:

```python
svg = food_overlay(fy)
myTerr.hexGrid.builder.adjust("food", svg)
myTerr.hexGrid.builder.show()
```

Skipping tier 0 lets the base map show through for barren land, so you only see dots where things actually grow.

In [ ]:
from HexMagic.terrainpatterns import TerrainPatterns

def food_overlay(fy: FoodYield, color: str = "#558B2F") -> str:
    """Build dotted overlay from computed food yield tiers."""
    terrain = fy.terrain
    grid = terrain.hexGrid
    
    patGen = TerrainPatterns(terrain)
    patterns = patGen.ballDensity(
        levels=fy.n_tiers,
        fills=[color],
        prefix="food_yield"
    )
    
    overlay = ""
    used = set()
    
    for i in range(len(terrain.elevations)):
        if terrain.elevations[i] <= 0:
            continue
        tier = int(fy.tiers[i])
        if tier == 0:
            continue  # Skip barren — let base map show through
        
        used.add(tier)
        pat_name = patterns[tier].attributes['id']
        hex_obj = grid.hexes[i]
        
        pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
        overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
    
    for tier in sorted(used):
        grid.builder.add_definition(patterns[tier])
    
    return overlay


In [ ]:
svg = food_overlay(fy)
myTerr.hexGrid.builder.layers = []
myTerr.terrainCream()
myTerr.hexGrid.builder.adjust("food", svg)
show(myTerr.hexGrid.builder)


I should make patterns unique

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, those duplicate definitions are piling up — `food_yield_1` through `food_yield_7` each appear 5 times. Quickest fix is to deduplicate in `food_overlay` before adding:

```python
for tier in sorted(used):
    pat = patterns[tier]
    pat_id = pat.attributes['id']
    # Only add if not already defined
    if not any(d.attributes.get('id') == pat_id for d in grid.builder.definitions):
        grid.builder.add_definition(pat)
```

Or if this is a recurring problem across your codebase, the cleaner fix is to make `add_definition` itself idempotent — deduplicate by ID in `SVGBuilder`:

```python
def add_definition(self, defn):
    pat_id = defn.attributes.get('id')
    if pat_id and any(d.attributes.get('id') == pat_id for d in self.definitions):
        return  # Already exists
    self.definitions.append(defn)
```

The second approach means you never have to think about it again from any overlay function.


# TerrainDisplay Overlay System

A declarative, composable system for rendering hex terrain with layered overlays. Inspired by FastHTML's component patterns — positional args are overlays, keyword args are context and HTML attributes.

## Quick Start

```python
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(),
    ClimateOverlay(),
    terrain=my_terrain,
    basins=my_basins,
    id="map"
)
```

That's it. Each overlay is a function that returns an `OverlaySpec`. `TerrainDisplay` renders them bottom-to-top by priority, wraps the result in a `Div(HexTouchMap(...))`, and passes through any HTML/HTMX attributes.

---

## How It Works

### The Three-Phase Render Pipeline

1. **Phase 1 — Run renderers**: Each overlay's `render(ctx)` runs in priority order. Overlays may mutate hex styles (fills, strokes) and/or return an SVG string.
2. **Phase 2 — Bake hex styles**: `grid.update()` is called once, flushing all hex style mutations into the `hexes` layer.
3. **Phase 3 — Stack SVG layers**: Any non-empty SVG strings are added as named layers on top of the hexes, in priority order.

### The Contract

Overlay renderers **may**:
- Mutate hex styles on `ctx.terrain` / `ctx.grid` (e.g. setting `grid.hexes[i].style`)
- Call `ctx.builder.add_definition()`, `ctx.builder.add_style()`, `ctx.builder.add_font()`
- Return an SVG string (or `""` for pure style mutations)

Overlay renderers **must not**:
- Call `ctx.builder.adjust()` — only `TerrainDisplay` does that

This keeps the builder under single ownership and makes overlay ordering predictable.

---

## Available Overlays

| Overlay | Priority | Type | Description |
|---------|----------|------|-------------|
| `TerrainOverlay()` | 5 | hex style | Elevation color bands (green→orange→red) |
| `CreamOverlay()` | 10 | hex style | Parchment tones by elevation + coast distance |
| `SoilOverlay()` | 15 | hex style | Hatch patterns by bedrock type (granite, basalt, etc.) |
| `ClimateOverlay(levels, min_density)` | 40 | SVG | Colored dots by climate zone + precipitation |
| `TemperatureOverlay(ocean_color)` | 41 | SVG | Temperature gradient icons |
| `FlowOverlay()` | 42 | SVG | Drainage flow direction arrows |
| `PrecipitationOverlay(levels, min_density, color)` | 43 | SVG | Rainfall density dots |
| `RiverOverlay(top_n, simplify_k, max_width)` | 60 | SVG | River network from drainage basins |

### Priority = Z-Order

Lower priority → drawn first → at the bottom. Higher priority → drawn last → on top. Think of it as a painter's algorithm: base coat first, fine details last.

---

## Composing Overlays

### Pick and choose

```python
# Minimal — just the parchment base
TerrainDisplay(CreamOverlay(), terrain=t)

# Add rivers (needs basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)

# Full detail
TerrainDisplay(
    CreamOverlay(),
    ClimateOverlay(levels=4),
    TemperatureOverlay(),
    RiverOverlay(top_n=12),
    terrain=t, basins=basins
)
```

### Order doesn't matter in the call

Priority controls render order, not argument position. These are equivalent:

```python
TerrainDisplay(RiverOverlay(), CreamOverlay(), terrain=t, basins=basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)
```

Both render cream (priority 10) before rivers (priority 60).

### Presets

Bundle common combinations into helper functions:

```python
def TerrainBase(terrain, **kw):
    return TerrainDisplay(CreamOverlay(), terrain=terrain, **kw)

def DetailedMap(terrain, basins, **kw):
    return TerrainDisplay(
        CreamOverlay(), ClimateOverlay(levels=4),
        RiverOverlay(), TemperatureOverlay(),
        terrain=terrain, basins=basins, **kw
    )
```

---

## Context & Dependencies

`TerrainDisplay` builds an `OverlayContext` from keyword args:

| Keyword | Stored in `ctx` | Enables |
|---------|-----------------|---------|
| `terrain=` | `ctx.terrain`, `ctx.grid`, `ctx.builder` | All overlays |
| `basins=` | `ctx.basins` | `RiverOverlay` |
| `soil=` | `ctx.soil` | `SoilOverlay` |
| `result=` | `ctx.result` (also auto-extracts `.basins`, `.soil`) | Overlays needing zoom result |
| `board=` | `ctx.board` | Borders, Names, Settlements |
| `c2f=` | `ctx.c2f` | Coarse-to-fine mapping for zoomed views |

Each `OverlaySpec` declares a `requires` set. If a required key is missing, the overlay is skipped with a warning — no crash.

```python
RiverOverlay()  # requires={'basins'}
# If basins=None, logs: "Skipping rivers: missing {'basins'}"
```

---

## Creating Custom Overlays

Any function that returns an `OverlaySpec` is an overlay:

```python
def HighPeakMarkers(threshold=2000, color="#FF4444", **kw) -> OverlaySpec:
    """Red circles on hexes above a given elevation."""
    def render(ctx: OverlayContext) -> str:
        parts = []
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > threshold:
                c = ctx.grid.hexes[i].center
                parts.append(f'<circle cx="{c.x}" cy="{c.y}" r="4" fill="{color}" opacity="0.7"/>')
        return '\n'.join(parts)
    return OverlaySpec("high_peaks", render, priority=55)

# Use it
TerrainDisplay(CreamOverlay(), HighPeakMarkers(threshold=1500), terrain=t)
```

### Custom overlay with hex style mutations

Return `""` and mutate styles instead:

```python
def OceanTintOverlay(color="#B3E5FC", **kw) -> OverlaySpec:
    """Recolor all ocean hexes."""
    def render(ctx: OverlayContext) -> str:
        style = StyleCSS("ocean_tint", fill=color)
        ctx.builder.add_style(style)
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev <= 0:
                ctx.grid.hexes[i].style = style
        return ""
    return OverlaySpec("ocean_tint", render, priority=8)
```

### Custom overlay with dependencies

```python
def BasinLabelOverlay(**kw) -> OverlaySpec:
    """Label each drainage basin at its mouth."""
    def render(ctx: OverlayContext) -> str:
        # ctx.basins is guaranteed present because of requires
        labels = []
        for basin in ctx.basins.top_basins(5):
            mouth = ctx.grid.hexes[basin.mouth].center
            labels.append(f'<text x="{mouth.x}" y="{mouth.y}" font-size="10">{basin.name}</text>')
        return '\n'.join(labels)
    return OverlaySpec("basin_labels", render, requires={'basins'}, priority=65)
```

---

## HTMX & HTML Attributes

All `**kwargs` that aren't recognized context keys pass through to the outer `Div`:

```python
TerrainDisplay(
    CreamOverlay(), RiverOverlay(),
    terrain=t, basins=basins,
    id="map",
    cls="w-full h-full",
    hx_swap="outerHTML",
    hx_get="/refresh_map"
)
# Produces: Div(HexTouchMap(...), id="map", cls="w-full h-full", hx_swap="outerHTML", ...)
```

Use `map_cls=` to set the class on the inner `HexTouchMap` (default: `"w-full h-full"`).

---

## Debug Mode

Pass `debug=True` to get the builder's diagnostic view instead of the rendered map:

```python
TerrainDisplay(CreamOverlay(), SoilOverlay(), terrain=t, soil=soil, debug=True)
```

This shows definitions, styles, and layers with their sizes — useful for checking that overlays registered correctly without side-effecting the builder.

---

## Route Integration Example

```python
@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    active = get_active_game(session)
    settlement = get_settlement(id)
    result = active.cover.zoom_region_fast(settlement.region(active.grid, rings=5))

    overlays = session_overlays(session)  # e.g. {'cream', 'rivers', 'climate'}
    components = overlay_set_to_components(overlays)

    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map",
        hx_swap="outerHTML"
    )

def overlay_set_to_components(names: set) -> list:
    """Convert a set of overlay names to component instances."""
    registry = {
        'cream': CreamOverlay,
        'soil': SoilOverlay,
        'climate': ClimateOverlay,
        'temperature': TemperatureOverlay,
        'rivers': RiverOverlay,
        'flow': FlowOverlay,
        'precipitation': PrecipitationOverlay,
    }
    return [registry[k]() for k in names if k in registry]
```

---

## Summary

| Concept | Pattern |
|---------|---------|
| Add an overlay | Pass `SomeOverlay()` as a positional arg |
| Configure an overlay | Pass params: `ClimateOverlay(levels=4)` |
| Set render order | Set `priority` in `OverlaySpec` (low = bottom) |
| Declare dependencies | Set `requires={'basins'}` in `OverlaySpec` |
| Pass HTML attrs | Use keyword args: `id=`, `cls=`, `hx_swap=` |
| Debug | `debug=True` |
| Create custom | Write a function returning `OverlaySpec` |
```

Lets make a FoodOverlay in the new overlayStyle system

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's a `FoodOverlay` following the system's conventions:

```python
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)
```

Usage:

```python
TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(),
    terrain=myTerr,
    basins=basins,
)
```

Or tuned:

```python
TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)
```

Priority 44 puts it right between climate (40) and precipitation (43) overlays and below rivers (60), so rivers draw on top of the food dots. The `skip_zero=True` default lets barren terrain show the base map through.

In [ ]:
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


In [ ]:
TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)
